# March Machine Learning Mania 2026

|No| Name  | Notebook |
| --- | ------ | ------- |
| 00 | Dataset |  https://www.kaggle.com/competitions/march-machine-learning-mania-2026/data |
| 01 | Understand Data Structure  | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p1-dataset-overview-structure-understanding/ |
| 02 | Create a baseline model only on Male rqd. 4 datasets | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p2-basline-male-4-datasets/ |
| 03 | Features Eng | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p3-features-eng/ |
| | |  |

In [47]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

Features

- Part 2 V1 - Baseline model (`win diff`, `seed diff`)
- Part 3 
  - V1 - `margin diff`
  - V2 - `recent win pct diff` Hypothesis: Teams playing well recently perform better than tournament

# Import Data

In [48]:
# Root file
data_file_path = r"/kaggle/input/march-machine-learning-mania-2026"
# Root file
data_file_path = r"C:\Users\Rudra\Desktop\kaggle\NCAA\data"

# Load only required files
regular = pd.read_csv(os.path.join(data_file_path, "MRegularSeasonCompactResults.csv"))
tourney = pd.read_csv(os.path.join(data_file_path,"MNCAATourneyCompactResults.csv"))
seeds = pd.read_csv(os.path.join(data_file_path,"MNCAATourneySeeds.csv"))
submission = pd.read_csv(os.path.join(data_file_path,"SampleSubmissionStage1.csv"))

# Sample View
display(regular.sample(3))
display(tourney.sample(3))
display(seeds.sample(3))
display(submission.sample(3))

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
40114,1995,33,1276,83,1151,71,A,0
110744,2010,73,1293,92,1240,68,H,0
24591,1991,55,1112,99,1344,87,H,0


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
2003,2016,137,1218,77,1143,66,N,0
2470,2024,136,1450,66,1179,61,N,0
2005,2016,137,1268,79,1355,74,N,0


,Season,Seed,TeamID
1916,2014,Y06,1269
690,1995,Z03,1277
2609,2025,Y16b,1384


,ID,Pred
160424,2023_1216_1230,0.5
385243,2024_3403_3470,0.5
313917,2024_1337_1459,0.5


# Create Win Counts

In [49]:
# wins
wins = regular.groupby(['Season', 'WTeamID']).size().reset_index(name='wins')

# losses
losses = regular.groupby(['Season', 'LTeamID']).size().reset_index(name='losses')

# rename for merge
wins = wins.rename(columns={"WTeamID": 'TeamID'})
losses = losses.rename(columns={'LTeamID': 'TeamID'})

# Merge
team_stats = pd.merge(
    wins, losses,
    how="outer", on=["Season", 'TeamID']
).fillna(0)

# total games
team_stats["TotalGames"] = team_stats["wins"] + team_stats["losses"]

# Win %page
team_stats['WinPct'] = team_stats['wins'] / team_stats['TotalGames']

display(team_stats.sample(3))

,Season,TeamID,wins,losses,TotalGames,WinPct
8137,2011,1148,19.0,12.0,31.0,0.612903
13005,2024,1460,16.0,14.0,30.0,0.533333
11543,2020,1414,19.0,11.0,30.0,0.633333


# Clean Seeds

In [50]:
seeds['SeedNum'] = seeds['Seed'].str[1:3].astype(int)

display(seeds.sample(3))

,Season,Seed,TeamID,SeedNum
2181,2018,X13,1158,13
2381,2022,X11a,1323,11
1021,2000,Z14,1457,14


#  Prepare Tournament

In [51]:
# smaller team id -> team 1
tourney['Team1'] = tourney[['WTeamID', 'LTeamID']].min(axis=1)
# larger team id -> team 2
tourney['Team2'] = tourney[['WTeamID', 'LTeamID']].max(axis=1)

# if team1 won then 1 else 0
tourney['Team1Win'] = (tourney['WTeamID'] == tourney['Team1']).astype(int)

train = tourney[['Season', 'Team1', 'Team2', 'Team1Win']]
display(train.sample(3))
print(train.shape)

,Season,Team1,Team2,Team1Win
132,1987,1276,1298,1
1807,2013,1231,1241,1
1815,2013,1233,1326,0


(2585, 4)


#  Add Features

## Add Score Margin

- Win percentage does not tell HOW strong.
- Winning by 1 point vs 20 points is different.

In [52]:
regular['ScoreDiff'] = regular['WScore'] - regular['LScore']

# For Winners
w_margin = regular.groupby(['Season', 'WTeamID'])['ScoreDiff'].mean().reset_index()
w_margin = w_margin.rename(columns={'WTeamID':'TeamID', 'ScoreDiff':'AvgMargin'})

# For Losers
l_margin = regular.groupby(['Season','LTeamID'])['ScoreDiff'].mean().reset_index()
l_margin['ScoreDiff'] = -l_margin['ScoreDiff']
l_margin = l_margin.rename(columns={'LTeamID':'TeamID','ScoreDiff':'AvgMargin'})

# Combine
margin = pd.concat([w_margin, l_margin])
margin = margin.groupby(['Season','TeamID'])['AvgMargin'].mean().reset_index()
display(margin.sample(3))

,Season,TeamID,AvgMargin
11708,2021,1228,5.394928
460,1986,1331,3.138889
5039,2001,1412,1.277311


## Add Recent Win Pct

In [53]:
regular_recent =  regular[regular['DayNum'] > 100].copy()

# wins
wins_recent = regular_recent.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
wins_recent = wins_recent.rename(columns={'WTeamID': 'TeamID'})

# Losses
losses_recent = regular_recent.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')
losses_recent = losses_recent.rename(columns={'LTeamID': 'TeamID'})

recent_stats = pd.merge(
    wins_recent, losses_recent,
    on=['Season', 'TeamID'],
    how='outer'
).fillna(0)

recent_stats['Total'] = recent_stats['Wins'] + recent_stats['Losses']
recent_stats['RecentWinPct'] =  recent_stats['Wins'] / recent_stats['Total']

recent_stats.head(3)


,Season,TeamID,Wins,Losses,Total,RecentWinPct
0,1985,1102,3.0,5.0,8.0,0.375000
1,1985,1103,3.0,4.0,7.0,0.428571
2,1985,1104,7.0,2.0,9.0,0.777778


## Add Win Pct Diff & Add Seed Diff

In [54]:
def add_features(df:pd.date_range) -> pd.DataFrame:
    """Create features for both train and test(submission) data."""

    # Team1 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team1_WinPct'}).drop('TeamID', axis=1)

    # Team2 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team2_WinPct'}).drop('TeamID', axis=1)

    # team1 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team1_Seed'}).drop('TeamID', axis=1)

    # team 2 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team2_Seed'}).drop('TeamID', axis=1)

    # team 1 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team1_AvgMargin'}).drop('TeamID', axis=1)
    
    # team 2 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team2_AvgMargin'}).drop('TeamID', axis=1)
    
    # recent team 1 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team1_RecentWinPct'}).drop('TeamID', axis=1)

    # recent team 2 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team2_RecentWinPct'}).drop('TeamID', axis=1)


    return df

In [55]:
train = add_features(train)

# Create Final Model Features

In [56]:
def create_diff(df:pd.DataFrame) ->pd.DataFrame:
    """ Create Features difference."""
    df['WinPct_Diff'] = df['Team1_WinPct'] - df['Team2_WinPct']

    # lower seed = strong team so (Team2 - Team1)
    df['Seed_Diff'] = df['Team2_Seed'] - df['Team1_Seed'] 

    # Avg Team Margin Diff
    df['Margin_Diff'] = df['Team1_AvgMargin'] - df['Team2_AvgMargin']
    
    # Recent Win pct diff
    df['RecentWinPct_Diff'] = df['Team1_RecentWinPct'] - df['Team2_RecentWinPct']

    return df

In [57]:
train = create_diff(train)

train

,Season,Team1,Team2,Team1Win,Team1_WinPct,Team2_WinPct,Team1_Seed,Team2_Seed,Team1_AvgMargin,Team2_AvgMargin,Team1_RecentWinPct,Team2_RecentWinPct,WinPct_Diff,Seed_Diff,Margin_Diff,RecentWinPct_Diff
0,1985,1116,1234,1,0.636364,0.666667,9,8,1.125000,6.475000,0.555556,0.250000,-0.030303,-1,-5.350000,0.305556
1,1985,1120,1345,1,0.620690,0.680000,11,6,1.098485,-1.110294,0.636364,0.666667,-0.059310,-5,2.208779,-0.030303
2,1985,1207,1250,1,0.925926,0.379310,1,16,7.770000,-2.416667,1.000000,0.555556,0.546616,15,10.186667,0.444444
3,1985,1229,1425,1,0.740741,0.678571,9,8,0.960714,0.654971,0.625000,0.600000,0.062169,-1,0.305744,0.025000
4,1985,1242,1325,1,0.766667,0.740741,3,14,0.593168,0.614286,0.700000,0.777778,0.025926,11,-0.021118,-0.077778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2580,2025,1120,1277,1,0.848485,0.818182,1,2,5.682143,4.657407,0.666667,0.888889,0.030303,1,1.024735,-0.222222
2581,2025,1222,1397,1,0.882353,0.794118,1,2,7.400000,4.195767,1.000000,0.777778,0.088235,1,3.204233,0.222222
2582,2025,1120,1196,0,0.848485,0.882353,1,1,5.682143,5.700000,0.666667,0.900000,-0.033868,0,-0.017857,-0.233333
2583,2025,1181,1222,0,0.911765,0.882353,1,1,9.295699,7.400000,1.000000,1.000000,0.029412,0,1.895699,0.000000


# Baseline Model

In [58]:
train.columns

Index(['Season', 'Team1', 'Team2', 'Team1Win', 'Team1_WinPct', 'Team2_WinPct',
       'Team1_Seed', 'Team2_Seed', 'Team1_AvgMargin', 'Team2_AvgMargin',
       'Team1_RecentWinPct', 'Team2_RecentWinPct', 'WinPct_Diff', 'Seed_Diff',
       'Margin_Diff', 'RecentWinPct_Diff'],
      dtype='object')

In [59]:
X = train[['WinPct_Diff','Seed_Diff','Margin_Diff',  'RecentWinPct_Diff']]
y = train['Team1Win']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict_proba(X_val)[:, 1]

print(f'LogLoss {log_loss(y_val, preds)}')

LogLoss 0.5259381625999039


# Submission

In [60]:
def create_id(df):
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True)

    df['Season'] = df['Season'].astype(int)
    df['Team1'] = df['Team1'].astype(int)
    df['Team2'] = df['Team2'].astype(int)

    return df

submission = create_id(submission)
submission = add_features(submission)

# only men for now
submission_men = submission[
    (submission['Team1'] < 2000) &
    (submission['Team2'] < 2000)
].copy()

submission = create_diff(submission)

submission[['WinPct_Diff','Seed_Diff', 'Margin_Diff', 'RecentWinPct_Diff']] = \
submission[['WinPct_Diff','Seed_Diff', 'Margin_Diff', 'RecentWinPct_Diff']].fillna(0)

# Predict
submission['Pred'] = model.predict_proba(
    submission[['WinPct_Diff','Seed_Diff', 'Margin_Diff', 'RecentWinPct_Diff']]
)[:,1]

display(submission[['ID','Pred']])

# Export
submission[['ID','Pred']].to_csv("baseline_submission.csv", index=False)
print('submission.csv exported')

,ID,Pred
0,2022_1101_1102,0.629416
1,2022_1101_1103,0.491865
2,2022_1101_1104,0.491641
3,2022_1101_1105,0.618970
4,2022_1101_1106,0.606036
...,...,...
519139,2025_3477_3479,0.496207
519140,2025_3477_3480,0.496207
519141,2025_3478_3479,0.496207
519142,2025_3478_3480,0.496207


submission.csv exported
